# Util class for dynamic pricing agents

> Provides util functions for dynamic pricing agents

In [ ]:
#| default_exp agents.dynamic_pricing.utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import torch 
from scipy.optimize import root_scalar

In [ ]:
#| export
class GLMLink:
    """
    A class to represent a link function used in statistical models.
    Attributes
    ----------
    g : callable
        The link function g(x).
    g_inv : callable
        The inverse of the link function g⁻¹(x).
    g_prime : callable
        The derivative of the link function g'(x).
    link : str
        A string representing the type of link function.
    v : callable
        A function that computes the variance (derivative of the inverse link function).
    Methods
    -------
    __call__(x)
        Applies the link function g to the input x.
    """
    def __init__(self, g, g_inv, g_prime, link):
        self.g = g          # Link function g(x)
        self.g_inv = g_inv  # Inverse link function g⁻¹(x)
        self.g_prime = g_prime  # Derivative of the link function g'(x)
        self.link = link
        def v(x):
            return self.g_prime(self.g_inv(x))
        self.v = v
    def __call__(self, x):
        return self.g(x)

In [ ]:
#| export
def get_price_function(function_form="linear"):
    if function_form == "linear":
        def price_function(x, alpha, beta):
            assert len(alpha) == len(x) and len(beta) == len(x)
            return np.array(-np.divide(np.dot(alpha, x), 2*np.dot(beta, x) + 1e-10))
        return price_function
    
    if function_form == "logit":
        
        def price_function(x, alpha, beta):
            assert len(alpha) == len(x) and len(beta) == len(x)
            M = 3.5
            a = np.dot(alpha, x)
            b = -np.dot(beta, x)
                
            def sigma(z):
                return 1 / (1 + np.exp(-z))
            def revenue_derivative(p):
                z = a - b*p
                s = sigma(z)
                return M * (s - b * p * (1 - s) * s)
            p_guess = 1.0 / b
            if p_guess <= 0:
                return np.array(0.0)
            bracket = [max(1e-4, p_guess * 0.1), p_guess * 10]

            # Ensure f(a) and f(b) have different signs
            f_a = revenue_derivative(bracket[0])
            f_b = revenue_derivative(bracket[1])
            if f_a * f_b > 0:
                return np.array(p_guess)

            result = root_scalar(revenue_derivative, bracket=bracket, method='brentq', xtol=1e-8, maxiter=100)

            if not result.converged:
                raise RuntimeError(f"Logit optimal price solver did not converge for a={a}, b={b}, M={M}")

            return np.array(result.root)
        
        return price_function
    
    if function_form == "exponential":
        def price_function(x, alpha, beta):
            assert len(alpha) == len(x) and len(beta) == len(x)
            return np.array(-np.divide(1, np.dot(beta, x)+ 1e-10))
        return price_function
    

In [ ]:
alpha = np.array([5.17324797588539])
beta = np.array([-1.815570567389732])
x = np.array([1])
linear_price_function = get_price_function("linear")
print("Linear Price Function Output:", linear_price_function(x, alpha, beta))
logit_price_function = get_price_function("logit")
print("Logit Price Function Output:", logit_price_function(x, alpha, beta))
exponential_price_function = get_price_function("exponential")
print("Exponential Price Function Output:", exponential_price_function(x, alpha, beta))

Linear Price Function Output: 1.4246893150139301
Logit Price Function Output: 2.23406701385967
Exponential Price Function Output: 0.550791039476252
